# Demostración del Deliverable 2 con Phi-3.5-mini

Corre el baseline y el sistema sobre los casos 40 a 49 en una T4 de Colab. Las celdas se
ejecutan en orden. La primera clona el repositorio, así que no hay que subir archivos.

Para elegir la GPU: *Entorno de ejecución*, *Cambiar tipo de entorno de ejecución*, T4 GPU.


In [ ]:
%cd /content
!nvidia-smi -L
!pip -q install transformers==5.17.0 accelerate bitsandbytes
!rm -rf /content/proyecto && git clone -q https://github.com/Luquitas02/genai-inscripcion-ramos /content/proyecto
!git -C /content/proyecto log -1 --format='commit %h  %ad  %s'
import torch; print('cuda:', torch.cuda.is_available())
# transformers queda fijo en 5.17.0, la version con que se midio del 17 al 19 de septiembre.
# Las demas quedan impresas para poder registrarlas en requirements-colab.txt.
from importlib.metadata import version
for paquete in ('torch', 'transformers', 'accelerate', 'bitsandbytes'):
    print(f'{paquete}=={version(paquete)}')

## Autopruebas
Revisan el verificador, la ficha, el parseo y el encadenado de pasos. No usan la GPU.


In [ ]:
import subprocess, sys, os
os.chdir('/content/proyecto/scripts')
for args in (['verificador.py'], ['ficha.py'], ['pipeline.py'], ['runner_d2.py', '--pruebas']):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    ultima = (r.stdout.strip().splitlines() or ['(sin salida)'])[-1]
    print(f'[{"OK  " if r.returncode == 0 else "FALLA"}] {args[0]:16} {ultima[:60]}')
    if r.returncode != 0:
        raise SystemExit('*** NO SIGAS: fallo ' + args[0] + ' ***')
print('TODO EN ORDEN')

## Carga del modelo
Phi-3.5-mini en 4 bits. La carga tarda entre tres y cinco minutos.


In [ ]:
import sys
sys.path.insert(0, '/content/proyecto/scripts')
from runner import ModeloHF
import demo

modelo = ModeloHF('microsoft/Phi-3.5-mini-instruct', max_new_tokens=256)
print('modelo cargado')

## Demostración
Los diez primeros casos de nivel 3, del 40 al 49. El caso 40 es el caso de falla y se muestra
con sus tres pasos. Cada salida del modelo se compara con la corrida guardada en `resultados/`.


In [ ]:
casos_mostrados = demo.video(modelo)